# Run add-only dataframe column updates

Calls `scripts/update_dfs_add_columns.py` over a configurable list of subdirectories.

**What is added** (from `update_dfs.ipynb`, only if missing):
- `evt_*`: `topo_categ`, `genie_categ`, truth track dirs, `theta_mu_p` / `mc_theta_mu_p` (degrees), truth TKI as `mc_del_*`
- `mcnu_*`: `topo_categ`, `genie_categ`, `theta_mu_p` (degrees), TKI as `del_*` — **without** rewriting existing column names (no permanent `mc` prefix)
- other keys (`hdr_*`, `trk_*`, `split`, …): copied unchanged

**Safety**: write-aside → verify → publish to sibling `<dirname>_updated/` (same basenames). Originals are never replaced or opened for write. Manifest lives under the `_updated` directory.

In [29]:
from os import path
import shlex

# ---- configure (run from notebooks/ so ../../.. is cafpyana root) ----
CAFPYANA_ROOT = path.abspath("../../..")

PYTHON = path.join(CAFPYANA_ROOT, "envs/venv_py310_cafpyana/bin/python")
SCRIPT = path.join(
    CAFPYANA_ROOT,
    "analysis_village/numucc_1p0pi/scripts/update_dfs_add_columns.py",
)

# Directories that contain split HDF5 files (*.df / *.h5)
BASE_DIR = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs"
SUBDIRS = [
    "2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV"   
]
SUBDIRS = [path.join(BASE_DIR, d) for d in SUBDIRS]

# Optional central tmp tree for staging (prefer same filesystem as inputs).
# None → <each input dir>/update_dfs_tmp/
# Commit publishes to sibling <dirname>_updated/ (originals never replaced).
TMP_ROOT = None

DRY_RUN = False          # list files + planned *_updated destinations only
PROCESS_ONLY = False    # write+verify staging tmp; do not publish to *_updated
COMMIT_ONLY = False     # publish verified tmp from an existing manifest into *_updated
LIMIT = 0               # 0 = all; use a small N for a first test
VERBOSE = True          # DEBUG on console
# Verbose (DEBUG) log file; None to skip. Prefer a local/data path over pnfs.
# LOG_FILE = "/exp/sbnd/data/users/munjung/xsec/logs/update_dfs_run.log"
LOG_FILE = None

assert path.isfile(PYTHON), PYTHON
assert path.isfile(SCRIPT), SCRIPT
assert SUBDIRS, "Set SUBDIRS to one or more directories containing .df / .h5 files"
assert not (PROCESS_ONLY and COMMIT_ONLY)

cmd = [PYTHON, SCRIPT, "--dirs", *SUBDIRS]
if TMP_ROOT:
    cmd += ["--tmp-root", TMP_ROOT]
if DRY_RUN:
    cmd.append("--dry-run")
if PROCESS_ONLY:
    cmd.append("--process-only")
if COMMIT_ONLY:
    cmd.append("--commit-only")
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
if VERBOSE:
    cmd.append("-v")
if LOG_FILE:
    cmd += ["--log-file", LOG_FILE]

CMD_STR = " ".join(shlex.quote(c) for c in cmd)
print("CAFPYANA_ROOT:", CAFPYANA_ROOT)
print("SUBDIRS:", len(SUBDIRS))
print("LOG_FILE:", LOG_FILE)

CAFPYANA_ROOT: /exp/sbnd/app/users/munjung/xsec/freeze/cafpyana
SUBDIRS: 1


## Command (print only)

In [27]:
print(CMD_STR)

/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana/envs/venv_py310_cafpyana/bin/python /exp/sbnd/app/users/munjung/xsec/freeze/cafpyana/analysis_village/numucc_1p0pi/scripts/update_dfs_add_columns.py --dirs /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV --commit-only -v


## Run

Recommended first pass: `DRY_RUN=True`, then `LIMIT=1` + `PROCESS_ONLY=True`, inspect the staging tmp, then full run (`DRY_RUN=False`, `PROCESS_ONLY=False`) to publish into `<dirname>_updated/`, or `--commit-only` after a process-only pass.

Outputs land next to the input directory, e.g.  
`…/2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV_updated/`.

In [28]:
import subprocess

print("Running:\n", CMD_STR, flush=True)
proc = subprocess.run(cmd, cwd=CAFPYANA_ROOT)
print("exit code:", proc.returncode)
if proc.returncode != 0:
    raise SystemExit(proc.returncode)

Running:
 /exp/sbnd/app/users/munjung/xsec/freeze/cafpyana/envs/venv_py310_cafpyana/bin/python /exp/sbnd/app/users/munjung/xsec/freeze/cafpyana/analysis_village/numucc_1p0pi/scripts/update_dfs_add_columns.py --dirs /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV --commit-only -v


INFO dirs=['/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV']  files=6000  patterns=('*.df', '*.h5')
INFO   /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV/sel_all-mc-BNB_cosmics-detvar_CV_0.df  →  /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV_updated/sel_all-mc-BNB_cosmics-detvar_CV_0.df
INFO   /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV/sel_all-mc-BNB_cosmics-detvar_CV_0_matched.df  →  /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV_updated/sel_all-mc-BNB_cosmics-detvar_CV_0_matched.df
INFO   /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV/sel_all-mc-BNB_cosmics-detvar_CV_1.df  →  /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_09_03_032325__sel_all-mc-

exit code: 1


INFO   /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV/sel_all-mc-BNB_cosmics-detvar_CV_914.df  →  /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV_updated/sel_all-mc-BNB_cosmics-detvar_CV_914.df
INFO   /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV/sel_all-mc-BNB_cosmics-detvar_CV_914_matched.df  →  /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV_updated/sel_all-mc-BNB_cosmics-detvar_CV_914_matched.df
INFO   /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV/sel_all-mc-BNB_cosmics-detvar_CV_915.df  →  /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_09_03_032325__sel_all-mc-BNB_cosmics-detvar_CV_updated/sel_all-mc-BNB_cosmics-detvar_CV_915.df
INFO   /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_09_03_032325_

SystemExit: 1

/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
